# Contact Form Intake API

This lab deploys API Management and Service Bus. The API accepts contact form submissions and enqueues them to a Service Bus queue for asynchronous processing.


### 0 Initialize notebook variables

In [ ]:
import os
import sys
import json
import datetime
import hashlib

# Add shared utilities
sys.path.insert(1, '../../shared')
import utils

# Configuration
deployment_name = "contact-form"
resource_group_location = "westus2"

# Get subscription ID for unique naming
subscription_id = utils.get_current_subscription()
unique_suffix = hashlib.md5(subscription_id.encode()).hexdigest()[:8] if subscription_id else "default"

# Resource names
resource_group_name = f"rg-{deployment_name}"

print("Configuration:")
print(f"  Resource Group: {resource_group_name}")
print(f"  Location: {resource_group_location}")
print(f"  Unique Suffix: {unique_suffix}")


### 1 Verify Azure CLI and connected subscription

In [ ]:
# Verify Azure CLI
output = utils.run("az account show", "Azure CLI is configured", "Azure CLI not configured - run 'az login'")

if output.success and output.json_data:
    print(f"Subscription: {output.json_data['name']}")
    print(f"Subscription ID: {output.json_data['id']}")
    print(f"Tenant ID: {output.json_data['tenantId']}")


### 2 Create Resource Group

In [ ]:
# Create resource group
utils.create_resource_group(resource_group_name, resource_group_location)


### 3 Deploy Infrastructure using Bicep

In [ ]:
# Deploy infrastructure
outputs = utils.deploy_bicep(
    resource_group=resource_group_name,
    template_file="main.bicep",
    parameters={"location": resource_group_location}
)

if outputs:
    print("Deployed Resources:")
    for key, value in outputs.items():
        print(f"  {key}: {value['value'] if isinstance(value, dict) else value}")


### 4 Get Deployment Outputs

In [ ]:
# Get deployment outputs
outputs = utils.get_deployment_outputs(resource_group_name)

if outputs:
    apim_gateway_url = outputs.get('apimGatewayUrl', '')
    apim_name = outputs.get('apimName', '')
    service_bus_namespace_name = outputs.get('serviceBusNamespaceName', '')
    service_bus_queue_name = outputs.get('serviceBusQueueName', '')
    service_bus_queue_uri = outputs.get('serviceBusQueueUri', '')

    print(f"API Management Gateway: {apim_gateway_url}")
    print(f"Service Bus Namespace: {service_bus_namespace_name}")
    print(f"Service Bus Queue: {service_bus_queue_name}")
    print(f"Service Bus Queue URI: {service_bus_queue_uri}")
else:
    print("Failed to retrieve deployment outputs")


### 5 Get APIM Subscription Key

In [ ]:
# Get APIM subscription key
result = utils.run(
    f'az rest --method post --uri "https://management.azure.com/subscriptions/{subscription_id}/resourceGroups/{resource_group_name}/providers/Microsoft.ApiManagement/service/{apim_name}/subscriptions/master/listSecrets?api-version=2023-05-01-preview" --query primaryKey -o tsv',
    "Retrieved APIM subscription key",
    "Failed to get APIM subscription key"
)

if result.success:
    subscription_key = result.output
    print(f"Subscription Key: {subscription_key}")


### 6 Test the Contact API

In [ ]:
import requests
import uuid

# Test payload
payload = {
    "id": f"contact-{uuid.uuid4().hex[:8]}",
    "name": "Jane Doe",
    "email": "jane@example.com",
    "subject": "Question about pricing",
    "message": "Can you share your pricing tiers?",
    "submittedAt": datetime.datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ")
}

# API endpoint
api_url = f"{apim_gateway_url}/contact/submit"

# Headers
headers = {
    "Content-Type": "application/json",
    "Ocp-Apim-Subscription-Key": subscription_key
}

print(f"Sending request to: {api_url}")
print(f"Payload: {json.dumps(payload, indent=2)}")

# Make request
response = requests.post(api_url, json=payload, headers=headers)

print(f"Response Status: {response.status_code}")
try:
    print(f"Response Body: {json.dumps(response.json(), indent=2)}")
except Exception:
    print(f"Response Body: {response.text}")


### 7 Verify Queue Message Count

In [ ]:
# Check the number of active messages in the queue
result = utils.run(
    f'az servicebus queue show --resource-group {resource_group_name} --namespace-name {service_bus_namespace_name} --name {service_bus_queue_name} --query "countDetails.activeMessageCount" -o tsv',
    "Retrieved queue message count",
    "Failed to get queue message count"
)

if result.success:
    print(f"Active messages in queue: {result.output}")


### 8 Clean up resources

In [ ]:
# Uncomment to delete resources
# utils.delete_resource_group(resource_group_name)
print(f"To delete resources, run: az group delete --name {resource_group_name} --yes")
